# [15.1] LoRA, DoRA, and Adapter Controls - Exercises

Fill in the local PEFT control primitives, then run the visible tests.


In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path

import torch as t

chapter = "chapter15_peft_misalignment"
section = "part1_lora_dora_adapter_controls"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_lora_dora_adapter_controls.tests as tests

GT_TIER = "GT-0"
EXERCISE_ID = "15_1_lora_dora_adapter_controls"
EXPECTED_RUNTIME = "seconds for toy contracts; minutes for the CUDA trained LoRA and matched PEFT comparison"
REQUIRES_GPU = True


In [ ]:
@dataclass(frozen=True)
class AdapterDeltaReport:
    rank: int
    alpha: float
    update_norm: float
    nonzero_update: bool


@dataclass(frozen=True)
class DoRAWeightReport:
    target_norms: t.Tensor
    row_norms: t.Tensor
    max_norm_error: float
    norm_preserved: bool


@dataclass(frozen=True)
class IntruderDimensionReport:
    projection_fraction: float
    intruder_detected: bool


@dataclass(frozen=True)
class AdapterMechanismReport:
    accuracy_delta: float
    mechanism_delta: float
    accuracy_improved: bool
    mechanism_preserved: bool
    adapter_acceptable: bool


In [ ]:
def lora_delta(lora_a: t.Tensor, lora_b: t.Tensor, *, alpha: float = 1.0) -> t.Tensor:
    raise NotImplementedError()


tests.test_lora_delta_uses_scaled_b_matrix_times_a_matrix(lora_delta)


In [ ]:
def adapter_delta_report(
    lora_a: t.Tensor,
    lora_b: t.Tensor,
    *,
    alpha: float = 1.0,
    min_update_norm: float = 1e-6,
) -> AdapterDeltaReport:
    raise NotImplementedError()


tests.test_adapter_delta_report_records_rank_alpha_and_nonzero_update(adapter_delta_report)


In [ ]:
def dora_recompose_weight(
    base_weight: t.Tensor,
    adapter_delta: t.Tensor,
    magnitude: t.Tensor,
    *,
    eps: float = 1e-8,
) -> t.Tensor:
    raise NotImplementedError()


def dora_weight_report(
    base_weight: t.Tensor,
    adapter_delta: t.Tensor,
    magnitude: t.Tensor,
    *,
    max_allowed_norm_error: float = 1e-5,
) -> DoRAWeightReport:
    raise NotImplementedError()


tests.test_dora_recompose_weight_preserves_target_row_magnitudes(
    dora_recompose_weight,
    dora_weight_report,
)


In [ ]:
def intruder_dimension_report(
    adapter_delta: t.Tensor,
    protected_direction: t.Tensor,
    *,
    max_projection_fraction: float = 0.2,
) -> IntruderDimensionReport:
    raise NotImplementedError()


tests.test_intruder_dimension_report_measures_projection_fraction(intruder_dimension_report)


In [ ]:
def adapter_mechanism_report(
    *,
    adapter_accuracy: float,
    baseline_accuracy: float,
    adapter_mechanism_score: float,
    baseline_mechanism_score: float,
    min_accuracy_gain: float = 0.05,
    min_mechanism_delta: float = -0.02,
) -> AdapterMechanismReport:
    raise NotImplementedError()


tests.test_adapter_mechanism_report_requires_accuracy_and_mechanism(adapter_mechanism_report)


In [ ]:
def lora_smoke_test() -> dict:
    raise NotImplementedError()


def dora_smoke_test() -> dict:
    raise NotImplementedError()


def intruder_smoke_test() -> dict:
    raise NotImplementedError()


def mechanism_smoke_test() -> dict:
    raise NotImplementedError()


def run_smoke_test(cpu: bool = True) -> dict:
    raise NotImplementedError()


tests.test_smoke_wrappers_match_the_visible_contract(
    lora_smoke_test,
    dora_smoke_test,
    intruder_smoke_test,
    mechanism_smoke_test,
)
tests.test_notebook_contract(run_smoke_test)


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    tests.test_committed_verification_report_trained_peft_controls(report)
    gpu = report["metrics"]["gpu_test"]
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = run_gpu_test(max_vram_gb=24.0)
{key: gpu[key] for key in [
    "trained_lora_preflight_passed",
    "trained_lora_random_label_control_fails",
    "trained_lora_random_adapter_control_fails",
    "trained_lora_dora_norm_preserved",
    "matched_peft_comparison_passed",
    "matched_target_alignment_floor",
    "matched_max_distractor_abs_cosine",
    "peak_vram_gb",
]}
